# [15.1] LoRA, DoRA, and Adapter Controls - Solutions

Reference validation notebook for the PEFT adapter-control section.


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter15_peft_misalignment"
section = "part1_lora_dora_adapter_controls"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_lora_dora_adapter_controls.tests as tests
from chapter15_peft_misalignment.exercises.part1_lora_dora_adapter_controls import solutions


In [ ]:
tests.test_lora_delta_uses_scaled_b_matrix_times_a_matrix(solutions.lora_delta)
tests.test_adapter_delta_report_records_rank_alpha_and_nonzero_update(
    solutions.adapter_delta_report,
)
tests.test_dora_recompose_weight_preserves_target_row_magnitudes(
    solutions.dora_recompose_weight,
    solutions.dora_weight_report,
)
tests.test_intruder_dimension_report_measures_projection_fraction(
    solutions.intruder_dimension_report,
)
tests.test_adapter_mechanism_report_requires_accuracy_and_mechanism(
    solutions.adapter_mechanism_report,
)
tests.test_smoke_wrappers_match_the_visible_contract(
    solutions.lora_smoke_test,
    solutions.dora_smoke_test,
    solutions.intruder_smoke_test,
    solutions.mechanism_smoke_test,
)
tests.test_notebook_contract(solutions.run_smoke_test)


In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["lora"]["delta"] == [[6.0, 12.0], [8.0, 16.0]]
assert contract["lora"]["report"]["rank"] == 1
assert contract["lora"]["report"]["nonzero_update"]
assert contract["dora"]["row_norms"] == [10.0, 5.0]
assert contract["dora"]["norm_preserved"]
assert contract["intruder"]["intruder_detected"]
assert abs(contract["intruder"]["projection_fraction"] - 1.0) < 1e-6
assert contract["mechanism"]["adapter_acceptable"]
contract


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
expected = report["baselines"]["expected_metrics"]

assert report["notebook_id"] == "15_1_lora_dora_and_adapter_controls"
assert report["accepted"]
assert report["tests_passed"]
assert report["gt_tier"] == "GT-0"
assert gpu["cuda_available"]
assert gpu["device"] == report["gpu_name"]
assert gpu["adapter_nonzero_update"] is expected["adapter_nonzero_update"]
assert gpu["dora_norm_preserved"] is expected["dora_norm_preserved"]
assert gpu["intruder_detected"]
assert gpu["trained_lora_preflight_passed"] is expected["trained_lora_preflight_passed"]
assert gpu["trained_lora_adapter_accuracy"] >= expected["trained_lora_adapter_accuracy_min"]
assert gpu["trained_lora_baseline_accuracy"] <= expected["trained_lora_baseline_accuracy_max"]
assert gpu["trained_lora_random_label_accuracy"] <= expected["trained_lora_random_label_accuracy_max"]
assert gpu["trained_lora_random_adapter_accuracy"] <= expected["trained_lora_random_adapter_accuracy_max"]
assert gpu["trained_lora_merge_max_abs_diff"] <= expected["trained_lora_merge_max_abs_diff_max"]
assert gpu["trained_lora_adapter_rank"] <= expected["trained_lora_adapter_rank_max"]
assert gpu["trained_lora_target_direction_cosine"] >= expected["trained_lora_target_direction_cosine_min"]
assert gpu["trained_lora_random_label_control_fails"]
assert gpu["trained_lora_random_adapter_control_fails"]
assert gpu["trained_lora_dora_norm_preserved"] is expected["trained_lora_dora_norm_preserved"]
assert gpu["matched_peft_comparison_passed"] is expected["matched_peft_comparison_passed"]
assert gpu["matched_accuracy_floor"] >= expected["matched_accuracy_floor_min"]
assert gpu["matched_target_alignment_floor"] >= expected["matched_target_alignment_floor_min"]
assert gpu["matched_max_distractor_abs_cosine"] <= expected["matched_max_distractor_abs_cosine_max"]
assert gpu["matched_dora_norm_preserved"] is expected["matched_dora_norm_preserved"]
assert gpu["matched_lora_trainable_parameters"] == expected["matched_lora_trainable_parameters"]
assert gpu["matched_dora_trainable_parameters"] == expected["matched_dora_trainable_parameters"]
assert gpu["matched_full_finetune_trainable_parameters"] == expected["matched_full_finetune_trainable_parameters"]
assert gpu["within_vram_budget"]
assert report["peak_vram_gb"] <= expected["trained_lora_peak_vram_gb_max"]
{key: gpu[key] for key in [
    "device",
    "trained_lora_adapter_accuracy",
    "trained_lora_baseline_accuracy",
    "trained_lora_random_label_accuracy",
    "matched_accuracy_floor",
    "matched_target_alignment_floor",
    "peak_vram_gb",
]}
